Questo notebook fa parte di **Pensare in Python**, traduzione italiana non ufficiale di [*Think Python*, 3ª edizione](https://allendowney.github.io/ThinkPython/) di Allen B. Downey. Puoi leggere tutto il libro tradotto su <https://pensare-in-python.github.io/3a-edizione/>.

Le versioni stampata ed ebook dell'originale si possono acquistare da [Bookshop.org](https://bookshop.org/a/98697/9781098155438) e [Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325) (link di affiliazione dell'autore).

Se trovi utile questo libro, valuta di [sostenere i materiali didattici liberi e aperti](https://greenteapress.com/support/?book=tp3) di Green Tea Press.

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

# Dizionari

[Esegui questo notebook su Colab](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap10.ipynb)

Questo capitolo presenta un tipo predefinito chiamato dizionario.
È una delle migliori caratteristiche di Python – e il mattone su cui si costruiscono molti algoritmi efficienti ed eleganti.

Useremo i dizionari per calcolare il numero di parole distinte in un libro e il numero di volte in cui compare ciascuna.
E negli esercizi useremo i dizionari per risolvere giochi di parole.

## Un dizionario è una mappatura

Un **dizionario** (*dictionary*) è come una lista, ma più generale.
In una lista gli indici devono essere interi; in un dizionario possono essere (quasi) di qualsiasi tipo.
Per esempio, supponiamo di creare una lista di numeri scritti in lettere, così.

In [ ]:
lst = ['zero', 'one', 'two']

Possiamo usare un intero come indice per ottenere la parola corrispondente.

In [ ]:
lst[1]

Ma supponiamo di voler andare nella direzione opposta, e cercare una parola per ottenere l'intero corrispondente.
Non possiamo farlo con una lista, ma possiamo farlo con un dizionario.
Inizieremo creando un dizionario vuoto e assegnandolo a `numbers`.

In [ ]:
numbers = {}
numbers

Le parentesi graffe, `{}`, rappresentano un dizionario vuoto.
Per aggiungere elementi al dizionario, useremo le parentesi quadre.

In [ ]:
numbers['zero'] = 0

Questa assegnazione aggiunge al dizionario un **elemento** (*item*), che rappresenta l'associazione di una **chiave** (*key*) e di un **valore** (*value*).
In questo esempio, la chiave è la stringa `'zero'` e il valore è l'intero `0`.
Se visualizziamo il dizionario, vediamo che contiene un elemento, formato da una chiave e un valore separati da due punti, `:`.

In [ ]:
numbers

Possiamo aggiungere altri elementi così.

In [ ]:
numbers['one'] = 1
numbers['two'] = 2
numbers

Ora il dizionario contiene tre elementi.

Per cercare una chiave e ottenere il valore corrispondente, usiamo l'operatore parentesi quadre.

In [ ]:
numbers['two']

Se la chiave non è nel dizionario, otteniamo un `KeyError`.

In [ ]:
%%expect KeyError
numbers['three']


La funzione `len` funziona sui dizionari; restituisce il numero di elementi.

In [ ]:
len(numbers)

In linguaggio matematico, un dizionario rappresenta una **mappatura** (*mapping*) dalle chiavi ai valori, quindi si può anche dire che ogni chiave "corrisponde a" (*maps to*) un valore.
In questo esempio, ogni numero scritto in lettere corrisponde all'intero relativo.

La figura seguente mostra il diagramma di stato per `numbers`.

In [ ]:
from diagram import make_dict, Binding, Value

d1 = make_dict(numbers, dy=-0.3, offsetx=0.37)
binding1 = Binding(Value('numbers'), d1)

In [ ]:
from diagram import diagram, adjust, Bbox

width, height, x, y = [1.83, 1.24, 0.49, 0.85]
ax = diagram(width, height)
bbox = binding1.draw(ax, x, y)
# adjust(x, y, bbox)

Un dizionario è rappresentato da un riquadro con la parola "dict" all'esterno e gli elementi all'interno.
Ogni elemento è rappresentato da una chiave e una freccia che punta a un valore.
Gli apici indicano che qui le chiavi sono stringhe, non nomi di variabili.

## Creare dizionari

Nella sezione precedente abbiamo creato un dizionario vuoto e aggiunto gli elementi uno alla volta usando l'operatore parentesi quadre.
Avremmo potuto invece creare il dizionario tutto in una volta così.

In [ ]:
numbers = {'zero': 0, 'one': 1, 'two': 2}

Ogni elemento è formato da una chiave e un valore separati da due punti.
Gli elementi sono separati da virgole e racchiusi tra parentesi graffe.

Un altro modo per creare un dizionario è usare la funzione `dict`.
Possiamo creare un dizionario vuoto così.

In [ ]:
empty = dict()
empty

E possiamo fare una copia di un dizionario così.

In [ ]:
numbers_copy = dict(numbers)
numbers_copy

Spesso è utile fare una copia prima di eseguire operazioni che modificano i dizionari.

## L'operatore in

Anche l'operatore `in` funziona sui dizionari; ti dice se qualcosa compare come *chiave* nel dizionario.

In [ ]:
'one' in numbers

L'operatore `in` *non* verifica se qualcosa compare come valore.

In [ ]:
1 in numbers

Per vedere se qualcosa compare come valore in un dizionario, puoi usare il metodo `values`, che restituisce una sequenza di valori, e poi usare l'operatore `in`.

In [ ]:
1 in numbers.values()

Gli elementi di un dizionario Python sono memorizzati in una **tabella hash** (*hash table*), un modo di organizzare i dati che ha una proprietà notevole: l'operatore `in` impiega circa lo stesso tempo indipendentemente da quanti elementi ci sono nel dizionario.
Questo rende possibile scrivere algoritmi straordinariamente efficienti.

In [ ]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

Per dimostrarlo, confronteremo due algoritmi per trovare coppie di parole in cui una è l'inverso dell'altra – come `stressed` e `desserts`.
Inizieremo leggendo l'elenco di parole.

In [ ]:
word_list = open('words.txt').read().split()
len(word_list)

Ed ecco `reverse_word` del capitolo precedente.

In [ ]:
def reverse_word(word):
    return ''.join(reversed(word))

La funzione seguente scorre le parole della lista.
Per ciascuna, inverte le lettere e poi verifica se la parola invertita è nell'elenco.

In [ ]:
def too_slow():
    count = 0
    for word in word_list:
        if reverse_word(word) in word_list:
            count += 1
    return count

Questa funzione impiega più di un minuto per essere eseguita.
Il problema è che l'operatore `in` controlla le parole della lista una alla volta, a partire dall'inizio.
Se non trova ciò che cerca – cosa che succede la maggior parte delle volte – deve cercare fino alla fine.

Per misurare quanto tempo impiega una funzione, possiamo usare `%time`, che è uno dei "comandi magici predefiniti" di Jupyter.
Questi comandi non fanno parte del linguaggio Python, quindi potrebbero non funzionare in altri ambienti di sviluppo.

In [ ]:
# %time too_slow()

E l'operatore `in` è all'interno del ciclo, quindi viene eseguito una volta per ogni parola.
Poiché nella lista ci sono più di 100.000 parole, e per ciascuna controlliamo più di 100.000 parole, il numero totale di confronti è il quadrato del numero di parole – all'incirca – cioè quasi 13 miliardi.

In [ ]:
len(word_list)**2

Possiamo rendere questa funzione molto più veloce con un dizionario.
Il ciclo seguente crea un dizionario che contiene le parole come chiavi.

In [ ]:
word_dict = {}
for word in word_list:
    word_dict[word] = 1

I valori in `word_dict` sono tutti `1`, ma potrebbero essere qualsiasi cosa, perché non li cercheremo mai – useremo questo dizionario solo per verificare se una chiave esiste.

Ecco ora una versione della funzione precedente che sostituisce `word_list` con `word_dict`.

In [ ]:
def much_faster():
    count = 0
    for word in word_dict:
        if reverse_word(word) in word_dict:
            count += 1
    return count

Questa funzione impiega meno di un centesimo di secondo, quindi è circa 10.000 volte più veloce della versione precedente.

In generale, il tempo necessario per trovare un elemento in una lista è proporzionale alla lunghezza della lista.
Il tempo necessario per trovare una chiave in un dizionario è quasi costante – indipendentemente dal numero di elementi.

In [ ]:
%time much_faster()

## Una raccolta di contatori

Supponiamo che ti venga data una stringa e che tu voglia contare quante volte compare ogni lettera.
Un dizionario è uno strumento adatto a questo compito.
Inizieremo con un dizionario vuoto.

In [ ]:
counter = {}

Mentre scorriamo le lettere della stringa, supponiamo di vedere la lettera `'a'` per la prima volta.
Possiamo aggiungerla al dizionario così.

In [ ]:
counter['a'] = 1

Il valore `1` indica che abbiamo visto la lettera una volta.
In seguito, se vediamo di nuovo la stessa lettera, possiamo incrementare il contatore così.

In [ ]:
counter['a'] += 1

Ora il valore associato a `'a'` è `2`, perché abbiamo visto la lettera due volte.

In [ ]:
counter

La funzione seguente usa queste funzionalità per contare il numero di volte in cui ogni lettera compare in una stringa.

In [ ]:
def value_counts(string):
    counter = {}
    for letter in string:
        if letter not in counter:
            counter[letter] = 1
        else:
            counter[letter] += 1
    return counter

A ogni giro del ciclo, se `letter` non è nel dizionario, creiamo un nuovo elemento con chiave `letter` e valore `1`.
Se `letter` è già nel dizionario, incrementiamo il valore associato a `letter`.

Ecco un esempio.

In [ ]:
counter = value_counts('brontosaurus')
counter

Gli elementi di `counter` mostrano che la lettera `'b'` compare una volta, `'r'` compare due volte, e così via.

## Cicli e dizionari

Se usi un dizionario in un'istruzione `for`, questa scorre le chiavi del dizionario.
Per dimostrarlo, creiamo un dizionario che conta le lettere di `'banana'`.

In [ ]:
counter = value_counts('banana')
counter

Il ciclo seguente stampa le chiavi, che sono le lettere.

In [ ]:
for key in counter:
    print(key)

Per stampare i valori, possiamo usare il metodo `values`.

In [ ]:
for value in counter.values():
    print(value)

Per stampare chiavi e valori, possiamo scorrere le chiavi e cercare i valori corrispondenti.

In [ ]:
for key in counter:
    value = counter[key]
    print(key, value)

Nel prossimo capitolo vedremo un modo più conciso di fare la stessa cosa.

## Liste e dizionari

Puoi mettere una lista in un dizionario come valore.
Per esempio, ecco un dizionario che fa corrispondere al numero `4` una lista di quattro lettere.

In [ ]:
d = {4: ['r', 'o', 'u', 's']}
d

Ma non puoi mettere una lista in un dizionario come chiave.
Ecco cosa succede se ci proviamo.

In [ ]:
%%expect TypeError
letters = list('abcd')
d[letters] = 4

Ho detto prima che i dizionari usano tabelle hash, e questo significa che le chiavi devono essere **hashabili** (*hashable*).

Un **hash** è una funzione che riceve un valore (di qualsiasi tipo) e restituisce un intero.
I dizionari usano questi interi, chiamati valori hash, per memorizzare e cercare le chiavi.

Questo sistema funziona solo se una chiave è immutabile, così il suo valore hash è sempre lo stesso.
Ma se una chiave fosse mutabile, il suo valore hash potrebbe cambiare, e il dizionario non funzionerebbe.
Ecco perché le chiavi devono essere hashabili, e perché i tipi mutabili come le liste non lo sono.

Poiché i dizionari sono mutabili, nemmeno loro possono essere usati come chiavi.
Ma *possono* essere usati come valori.

## Accumulare una lista

In molti compiti di programmazione è utile scorrere una lista o un dizionario mentre se ne costruisce un altro.
Come esempio, scorreremo le parole di `word_dict` e creeremo una lista di palindromi – cioè parole che si leggono allo stesso modo in entrambi i sensi, come "noon" e "rotator".

Nel capitolo precedente, uno degli esercizi ti chiedeva di scrivere una funzione che verifica se una parola è un palindromo.
Ecco una soluzione che usa `reverse_word`.

In [ ]:
def is_palindrome(word):
    """Check if a word is a palindrome."""
    return reverse_word(word) == word

Se scorriamo le parole di `word_dict`, possiamo contare il numero di palindromi così.

In [ ]:
count = 0

for word in word_dict:
    if is_palindrome(word):
        count +=1
        
count

Ormai questo schema è familiare.

* Prima del ciclo, `count` viene inizializzato a `0`.

* All'interno del ciclo, se `word` è un palindromo, incrementiamo `count`.

* Quando il ciclo termina, `count` contiene il numero totale di palindromi.

Possiamo usare uno schema simile per creare una lista di palindromi.

In [ ]:
palindromes = []

for word in word_dict:
    if is_palindrome(word):
        palindromes.append(word)

palindromes[:10]

Ecco come funziona:

* Prima del ciclo, `palindromes` viene inizializzata con una lista vuota.

* All'interno del ciclo, se `word` è un palindromo, lo aggiungiamo in fondo a `palindromes`.

* Quando il ciclo termina, `palindromes` è una lista di palindromi.

In questo ciclo, `palindromes` è usata come **accumulatore** (*accumulator*), cioè una variabile che raccoglie o accumula dati durante un calcolo.

Supponiamo ora di voler selezionare solo i palindromi con sette o più lettere.
Possiamo scorrere `palindromes` e creare una nuova lista che contiene solo i palindromi lunghi.

In [ ]:
long_palindromes = []

for word in palindromes:
    if len(word) >= 7:
        long_palindromes.append(word)
        
long_palindromes

Scorrere una lista in questo modo, selezionando alcuni elementi e scartandone altri, si chiama **filtraggio** (*filtering*).

## Memo

Se hai eseguito la funzione `fibonacci` del [capitolo 6](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap06.ipynb), forse hai notato che più grande è l'argomento che fornisci, più tempo impiega la funzione.

In [ ]:
def fibonacci(n):
    if n == 0:
        return 0
    
    if n == 1:
        return 1

    return fibonacci(n-1) + fibonacci(n-2)

Inoltre, il tempo di esecuzione aumenta rapidamente.
Per capire perché, considera la figura seguente, che mostra il **grafo delle chiamate** (*call graph*) di
`fibonacci` con `n=4`:

In [ ]:
from diagram import make_binding, Frame, Arrow

bindings = [make_binding('n', i) for i in range(5)]
frames = [Frame([binding]) for binding in bindings]

In [ ]:
arrowprops = dict(arrowstyle="-", color='gray', alpha=0.5, ls='-', lw=0.5)

def left_arrow(ax, bbox1, bbox2):
    x = bbox1.xmin + 0.1
    y = bbox1.ymin
    dx = bbox2.xmax - x - 0.1
    dy = bbox2.ymax - y
    arrow = Arrow(dx=dx, dy=dy, arrowprops=arrowprops)
    return arrow.draw(ax, x, y)

def right_arrow(ax, bbox1, bbox2):
    x = bbox1.xmax - 0.1
    y = bbox1.ymin
    dx = bbox2.xmin - x + 0.1
    dy = bbox2.ymax - y
    arrow = Arrow(dx=dx, dy=dy, arrowprops=arrowprops)
    return arrow.draw(ax, x, y)

In [ ]:
from diagram import diagram, adjust, Bbox

width, height, x, y = [4.94, 2.16, -1.03, 1.91]
ax = diagram(width, height)

dx = 0.6
dy = 0.55

bboxes = []
bboxes.append(frames[4].draw(ax, x+6*dx, y))

bboxes.append(frames[3].draw(ax, x+4*dx, y-dy))
bboxes.append(frames[2].draw(ax, x+8*dx, y-dy))

bboxes.append(frames[2].draw(ax, x+3*dx, y-2*dy))
bboxes.append(frames[1].draw(ax, x+5*dx, y-2*dy))
bboxes.append(frames[1].draw(ax, x+7*dx, y-2*dy))
bboxes.append(frames[0].draw(ax, x+9*dx, y-2*dy))

bboxes.append(frames[1].draw(ax, x+2*dx, y-3*dy))
bboxes.append(frames[0].draw(ax, x+4*dx, y-3*dy))

left_arrow(ax, bboxes[0], bboxes[1])
left_arrow(ax, bboxes[1], bboxes[3])
left_arrow(ax, bboxes[3], bboxes[7])
left_arrow(ax, bboxes[2], bboxes[5])

right_arrow(ax, bboxes[0], bboxes[2])
right_arrow(ax, bboxes[1], bboxes[4])
right_arrow(ax, bboxes[2], bboxes[6])
right_arrow(ax, bboxes[3], bboxes[8])

bbox = Bbox.union(bboxes)
# adjust(x, y, bbox)

Un grafo delle chiamate mostra un insieme di frame di funzione, con linee che collegano ogni frame ai frame delle funzioni che chiama.
In cima al grafo, `fibonacci` con `n=4` chiama `fibonacci` con `n=3` e `n=2`.
A sua volta, `fibonacci` con `n=3` chiama `fibonacci` con `n=2` e `n=1`. E così via.

Conta quante volte vengono chiamate `fibonacci(0)` e `fibonacci(1)`.
È una soluzione inefficiente del problema, e peggiora man mano che l'argomento cresce.

Una soluzione è tenere traccia dei valori già calcolati memorizzandoli in un dizionario.
Un valore calcolato in precedenza e memorizzato per un uso successivo si chiama **memo**.
Ecco una versione "memoizzata" di `fibonacci`:

In [ ]:
known = {0:0, 1:1}

def fibonacci_memo(n):
    if n in known:
        return known[n]

    res = fibonacci_memo(n-1) + fibonacci_memo(n-2)
    known[n] = res
    return res

`known` è un dizionario che tiene traccia dei numeri di Fibonacci che conosciamo già.
Inizia con due elementi: `0` corrisponde a `0` e `1` corrisponde a `1`.

Ogni volta che `fibonacci_memo` viene chiamata, controlla `known`.
Se il risultato è già lì, può restituirlo immediatamente.
Altrimenti deve calcolare il nuovo valore, aggiungerlo al dizionario e restituirlo.

Confrontando le due funzioni, `fibonacci(40)` impiega circa 30 secondi.
`fibonacci_memo(40)` impiega circa 30 microsecondi, quindi è un milione di volte più veloce.
Nel notebook di questo capitolo vedrai da dove vengono queste misurazioni.

In [ ]:
# %time fibonacci(40)

In [ ]:
%time fibonacci_memo(40)

## Debugging

Man mano che lavori con insiemi di dati più grandi, fare debugging stampando e controllando l'output a mano può diventare scomodo. Ecco alcuni suggerimenti per il debugging di grandi insiemi di dati:

1.  Riduci l'input: se possibile, riduci la dimensione dei dati. Per esempio, se il
    programma legge un file di testo, inizia con le sole prime 10 righe, o
    con l'esempio più piccolo che riesci a trovare. Puoi modificare i
    file stessi oppure (meglio) modificare il programma in modo che legga solo
    le prime `n` righe.

    Se c'è un errore, puoi ridurre `n` al valore più piccolo per cui l'errore si verifica.
    Man mano che trovi e correggi gli errori, puoi aumentare `n` gradualmente.


2. Controlla riepiloghi e tipi: invece di stampare e controllare l'intero insieme di dati, valuta di
    stampare dei riepiloghi dei dati – per esempio, il numero di elementi di
    un dizionario o il totale di una lista di numeri.

    Una causa comune di errori di runtime è un valore che non è del tipo giusto. Per fare debugging di questo tipo di errore, spesso basta stampare il tipo di un valore.


3. Scrivi controlli automatici: a volte puoi scrivere codice che verifichi automaticamente gli errori. Per
    esempio, se stai calcolando la media di una lista di numeri, potresti
    verificare che il risultato non sia maggiore dell'elemento più grande
    della lista né minore del più piccolo. Questo si chiama "controllo di
    sanità" (*sanity check*) perché individua risultati "insensati".

    Un altro tipo di controllo confronta i risultati di due calcoli diversi per vedere se sono coerenti. Questo si chiama "controllo di coerenza" (*consistency check*).


4. Formatta l'output: formattare l'output di debugging può rendere più facile individuare un errore. Ne abbiamo visto un esempio nel [capitolo 6](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap06.ipynb). Un altro strumento che potresti trovare utile è il modulo `pprint`, che fornisce una funzione `pprint` che visualizza i tipi predefiniti in un formato più leggibile (`pprint` sta per "pretty print", stampa ordinata).

    Di nuovo, il tempo che dedichi a costruire un'impalcatura può ridurre il tempo che passi a fare debugging.


## Glossario

**dizionario** (*dictionary*):
Un oggetto che contiene coppie chiave-valore, chiamate anche elementi.

**elemento** (*item*):
In un dizionario, un altro nome per una coppia chiave-valore.

**chiave** (*key*):
Un oggetto che compare in un dizionario come prima parte di una coppia chiave-valore.

**valore** (*value*):
Un oggetto che compare in un dizionario come seconda parte di una coppia chiave-valore. È un uso più specifico rispetto a quello fatto finora della parola "valore".

**mappatura** (*mapping*):
Una relazione in cui ogni elemento di un insieme corrisponde a un elemento di un altro insieme.

**tabella hash** (*hash table*):
Una raccolta di coppie chiave-valore organizzata in modo da poter cercare una chiave e trovarne il valore in modo efficiente.

**hashabile** (*hashable*):
I tipi immutabili come interi, float e stringhe sono hashabili.
I tipi mutabili come liste e dizionari non lo sono.

**funzione hash** (*hash function*):
Una funzione che riceve un oggetto e calcola un intero che si usa per localizzare una chiave in una tabella hash.

**accumulatore** (*accumulator*):
Una variabile usata in un ciclo per sommare o accumulare un risultato.

**filtraggio** (*filtering*):
Scorrere una sequenza selezionando o scartando elementi.

**grafo delle chiamate** (*call graph*):
Un diagramma che mostra ogni frame creato durante l'esecuzione di un programma, con una freccia da ogni chiamante a ogni funzione chiamata.

**memo**:
Un valore calcolato e memorizzato per evitare calcoli futuri inutili.

## Esercizi

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Chiedi a un assistente virtuale

In questo capitolo ho detto che le chiavi di un dizionario devono essere hashabili e ne ho dato una breve spiegazione. Se vuoi maggiori dettagli, chiedi a un assistente virtuale "Perché le chiavi dei dizionari Python devono essere hashabili?"

In [una sezione precedente](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap10.ipynb) abbiamo memorizzato un elenco di parole come chiavi di un dizionario per poter usare una versione efficiente dell'operatore `in`.
Avremmo potuto fare la stessa cosa usando un `set` (insieme), che è un altro tipo di dato predefinito.
Chiedi a un assistente virtuale "Come creo un set Python da una lista di stringhe e verifico se una stringa è un elemento del set?"

### Esercizio

I dizionari hanno un metodo chiamato `get` che riceve una chiave e un valore predefinito.
Se la chiave compare nel dizionario, `get` restituisce il valore corrispondente; altrimenti restituisce il valore predefinito.
Per esempio, ecco un dizionario che fa corrispondere le lettere di una stringa al numero di volte in cui compaiono.

In [ ]:
counter = value_counts('brontosaurus')

Se cerchiamo una lettera che compare nella parola, `get` restituisce il numero di volte in cui compare.

In [ ]:
counter.get('b', 0)

Se cerchiamo una lettera che non compare, otteniamo il valore predefinito, `0`.

In [ ]:
counter.get('c', 0)

Usa `get` per scrivere una versione più concisa di `value_counts`.
Dovresti riuscire a eliminare l'istruzione `if`.

### Esercizio

Qual è la parola più lunga che ti viene in mente in cui ogni lettera compare una sola volta?
Vediamo se riusciamo a trovarne una più lunga di `unpredictably`.

Scrivi una funzione chiamata `has_duplicates` che riceve come parametro una sequenza – come una lista o una stringa – e restituisce `True` se c'è un elemento che compare nella sequenza più di una volta.

Per aiutarti a iniziare, ecco uno schema della funzione con dei doctest.

In [ ]:
def has_duplicates(t):
    """Check whether any element in a sequence appears more than once.
    
    >>> has_duplicates('banana')
    True
    >>> has_duplicates('ambidextrously')
    False
    >>> has_duplicates([1, 2, 2])
    True
    >>> has_duplicates([1, 2, 3])
    False
    """
    return None

In [ ]:
# Solution goes here

Puoi usare `doctest` per testare la tua funzione.

In [ ]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

run_doctests(has_duplicates)

Puoi usare questo ciclo per trovare le parole più lunghe senza lettere ripetute.

In [ ]:
no_repeats = []

for word in word_list:
    if len(word) > 12 and not has_duplicates(word):
        no_repeats.append(word)
        
no_repeats

### Esercizio

Scrivi una funzione chiamata `find_repeats` che riceve un dizionario che fa corrispondere a ogni chiave un contatore, come il risultato di `value_counts`.
Deve scorrere il dizionario e restituire una lista delle chiavi che hanno un conteggio maggiore di `1`.
Puoi usare lo schema seguente per iniziare.

In [ ]:
def find_repeats(counter):
    """Makes a list of keys with values greater than 1.
    
    counter: dictionary that maps from keys to counts
    
    returns: list of keys
    """
    return []

In [ ]:
# Solution goes here

Puoi usare gli esempi seguenti per testare il tuo codice.
Per prima cosa, creeremo un dizionario che fa corrispondere le lettere ai conteggi.

In [ ]:
counter1 = value_counts('banana')
counter1

Il risultato di `find_repeats` dovrebbe essere `['a', 'n']`.

In [ ]:
repeats = find_repeats(counter1)
repeats

Ecco un altro esempio che parte da una lista di numeri.
Il risultato dovrebbe essere `[1, 2]`.

In [ ]:
counter1 = value_counts([1, 2, 3, 2, 1])
repeats = find_repeats(counter1)
repeats

### Esercizio

Supponiamo di eseguire `value_counts` con due parole diverse e di salvare i risultati in due dizionari.

In [ ]:
counter1 = value_counts('brontosaurus')
counter2 = value_counts('apatosaurus')

Ogni dizionario fa corrispondere un insieme di lettere al numero di volte in cui compaiono.
Scrivi una funzione chiamata `add_counters` che riceve due dizionari come questi e restituisce un nuovo dizionario che contiene tutte le lettere e il numero totale di volte in cui compaiono nell'una o nell'altra parola.

Ci sono molti modi per risolvere questo problema.
Una volta che hai una soluzione funzionante, valuta di chiedere a un assistente virtuale soluzioni diverse.

In [ ]:
# Solution goes here

In [ ]:
# Solution goes here

### Esercizio

Una parola è "intrecciata" (*interlocking*) se possiamo dividerla in due parole prendendo le lettere alternate.
Per esempio, "schooled" è una parola intrecciata perché può essere divisa in "shoe" e "cold".

Per selezionare le lettere alternate di una stringa, puoi usare un operatore di slice con tre componenti che indicano dove iniziare, dove fermarsi e il "passo" (*step*) tra le lettere.

Nella slice seguente, il primo componente è `0`, quindi partiamo dalla prima lettera.
Il secondo componente è `None`, il che significa che dobbiamo arrivare fino alla fine della stringa.
E il terzo componente è `2`, quindi ci sono due passi tra le lettere che selezioniamo.

In [ ]:
word = 'schooled'
first = word[0:None:2]
first

Invece di fornire `None` come secondo componente, possiamo ottenere lo stesso effetto omettendolo del tutto.
Per esempio, la slice seguente seleziona le lettere alternate a partire dalla seconda.

In [ ]:
second = word[1::2]
second

Scrivi una funzione chiamata `is_interlocking` che riceve una parola come argomento e restituisce `True` se può essere divisa in due parole intrecciate.

In [ ]:
# Solution goes here

Puoi usare il ciclo seguente per trovare le parole intrecciate nell'elenco.

In [ ]:
for word in word_list:
    if len(word) >= 8 and is_interlocking(word):
        first = word[0::2]
        second = word[1::2]
        print(word, first, second)

[Pensare in Python](https://pensare-in-python.github.io/3a-edizione/) – traduzione italiana non ufficiale di [Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Licenza del codice: [MIT License](https://mit-license.org/)

Licenza del testo (originale e traduzione): [Creative Commons Attribuzione - Non commerciale - Condividi allo stesso modo 4.0 Internazionale](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.it)